In [ ]:
%pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [8]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "ubarsc/python-fmask",
    "alpscore_alpscore",
    "cnes_cars",
    "chembl_fpsim2",
    "sinzlab_nndichromacy",
    "mivion_swisseph",
    "niklastr_promise",
    "activitymonitoring_biobankaccelerometeranalysis",
    "nasa-pds_harvest",
    "bluescarni_mppp",
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00489059106ca30896d323bbd07cb9600f50e890,ubarsc_python-fmask


In [9]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 58%|███████████████████▏             | 1056/1822 [18:14<13:12,  1.03s/it]

Got Errors {'09acbf181f7caff991e19b024bf6b86df7b6e9c3': 'Key 09acbf181f7caff991e19b024bf6b86df7b6e9c3 not found in /da5_fast/All.sha1c/commit_9.tch'}


 58%|███████████████████▏             | 1057/1822 [18:15<13:15,  1.04s/it]

Got Errors {'0fc33634fa47634a4096fd4d031a9425cbe6a59e': 'Key 0fc33634fa47634a4096fd4d031a9425cbe6a59e not found in /da5_fast/All.sha1c/commit_15.tch'}


 58%|███████████████████▏             | 1062/1822 [18:21<13:06,  1.03s/it]

...

Got Errors {'3b4d83cab22317815e5baa6cd02d4a2c403feef7': 'Key 3b4d83cab22317815e5baa6cd02d4a2c403feef7 not found in /da5_fast/All.sha1c/commit_59.tch'}


 88%|████████████████████████████▉    | 1601/1822 [27:40<03:47,  1.03s/it]

Got Errors {'6aa4ed0ea6ab81711f2a679515fa63b0c722d6f9': 'Key 6aa4ed0ea6ab81711f2a679515fa63b0c722d6f9 not found in /da5_fast/All.sha1c/commit_106.tch'}


100%|█████████████████████████████████| 1822/1822 [31:28<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00489059106ca30896d323bbd07cb9600f50e890,3f4e33a0ff6ce3ac6019158e019c79e6fe215cbf,[72e6f7c6591003735f725209951a64f9751f6d63],Neil Flood <neilflood@fastmail.fm>,1643078770,+1000,Neil Flood <neilflood@fastmail.fm>,1643078770,+1000,fillminima.py flake8 errors\n,00489059106ca30896d323bbd07cb9600f50e890,ubarsc_python-fmask
1,01ce91861d7887fd9954ba3646ec4ce8d6d7a796,bcf33d693b12b09f2ee1630645b994cdc84a9800,[5a97e57c9fcf1cc25614bd0bf38897865bd46f7f],Sam Gillingham <gillingham.sam@gmail.com>,1461733841,+1000,Sam Gillingham <gillingham.sam@gmail.com>,1461733841,+1000,fixed warning on OSX where I was treating an i...,01ce91861d7887fd9954ba3646ec4ce8d6d7a796,ubarsc_python-fmask


In [10]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00489059106ca30896d323bbd07cb9600f50e890', 'tree': '3f4e33a0ff6ce3ac6019158e019c79e6fe215cbf', 'parent': ['72e6f7c6591003735f725209951a64f9751f6d63'], 'author': 'Neil Flood <neilflood@fastmail.fm>', 'author_time': 1643078770, 'author_tz': '+1000', 'committer': 'Neil Flood <neilflood@fastmail.fm>', 'committer_time': 1643078770, 'committer_tz': '+1000', 'message': 'fillminima.py flake8 errors\n'}


In [12]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [16]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,bluescarni_mppp,00489059106ca30896d323bbd07cb9600f50e890,Neil Flood <neilflood@fastmail.fm>,1643078770,fillminima.py flake8 errors\n


In [14]:
#mode a means append, so you have all your projects in the same file
yournetid='pmarino'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [23]:
import requests
from IPython.display import Markdown, display

github_projects = [
    "ubarsc/python-fmask",
    "ALPSCore/ALPSCore",
    "CNES/cars",
    "chembl/FPSim2",
    "sinzlab/nndichromacy",
    "mivion/swisseph",
    "niklastr/promise",
    "activityMonitoring/biobankAccelerometerAnalysis",
    "NASA-PDS/harvest",
    "bluescarni/mppp"
]

rows = []

for repo in github_projects:
    repo_url = f"https://api.github.com/repos/{repo}"
    r = requests.get(repo_url)

    if r.status_code != 200:
        print(f"ERROR: {repo}")
        print(f"Status: {r.status_code}")
        print(r.json())
        print()
        continue

    repo_data = r.json()

    commits_url = f"https://api.github.com/repos/{repo}/commits?per_page=1"
    c = requests.get(commits_url)

    if c.status_code == 200 and len(c.json()) > 0:
        last_commit = c.json()[0]["commit"]["committer"]["date"][:10]
    else:
        last_commit = "N/A"

    rows.append([
        repo,
        repo_data["stargazers_count"],
        repo_data["forks_count"],
        last_commit
    ])

table = "| Project | Stars | Forks | Last Commit Date |\n"
table += "|---|---:|---:|---|\n"

for repo, stars, forks, last_commit in rows:
    table += f"| {repo} | {stars} | {forks} | {last_commit} |\n"

display(Markdown(table))

| Project | Stars | Forks | Last Commit Date |
|---|---:|---:|---|
| ubarsc/python-fmask | 79 | 24 | 2026-02-24 |
| ALPSCore/ALPSCore | 114 | 46 | 2026-09-25 |
| CNES/cars | 398 | 48 | 2026-09-18 |
| chembl/FPSim2 | 184 | 24 | 2026-02-26 |
| sinzlab/nndichromacy | 8 | 7 | 2024-06-10 |
| mivion/swisseph | 238 | 87 | 2025-05-13 |
| niklastr/promise | 0 | 0 | 2025-04-21 |
| activityMonitoring/biobankAccelerometerAnalysis | 251 | 71 | 2025-11-10 |
| NASA-PDS/harvest | 6 | 3 | 2026-09-21 |
| bluescarni/mppp | 321 | 29 | 2024-12-10 |


In [24]:
from IPython.display import Markdown, display

# Convert WoC Unix timestamps to readable dates
df_commit_data['datetime'] = pd.to_datetime(
    df_commit_data['author_time'],
    unit='s',
    utc=True
)

# Number of commits from the original WoC project-to-commit results
commit_counts = (
    df.groupby('project')
      .size()
      .rename('Commits')
)

# Authors and time range from detailed WoC commit data
details = (
    df_commit_data.groupby('project')
    .agg(
        Authors=('author', 'nunique'),
        Min_Time=('datetime', 'min'),
        Max_Time=('datetime', 'max')
    )
)

# Combine results
woc_summary = pd.concat([commit_counts, details], axis=1).reset_index()

# Only show the date/time cleanly
woc_summary['Min_Time'] = woc_summary['Min_Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
woc_summary['Max_Time'] = woc_summary['Max_Time'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Build Markdown table
table = "| Project | Commits | Authors | Max Time | Min Time |\n"
table += "|---|---:|---:|---|---|\n"

for _, row in woc_summary.iterrows():
    table += (
        f"| {row['project']} "
        f"| {row['Commits']} "
        f"| {row['Authors']} "
        f"| {row['Max_Time']} "
        f"| {row['Min_Time']} |\n"
    )

display(Markdown(table))

| Project | Commits | Authors | Max Time | Min Time |
|---|---:|---:|---|---|
| activitymonitoring_biobankaccelerometeranalysis | 1284 | 42 | 2025-10-31 10:05:05 | 2014-08-07 18:54:34 |
| alpscore_alpscore | 5025 | 75 | 2026-05-10 07:42:18 | 2003-05-06 17:17:45 |
| bluescarni_mppp | 3839 | 9 | 2024-12-10 11:20:09 | 2016-08-24 22:42:46 |
| chembl_fpsim2 | 934 | 13 | 2025-09-06 09:35:56 | 2018-10-22 00:17:20 |
| cnes_cars | 3748 | 100 | 2026-04-15 09:52:35 | 2020-07-08 12:50:21 |
| mivion_swisseph | 324 | 27 | 2025-05-13 07:06:30 | 2012-05-17 08:54:55 |
| nasa-pds_harvest | 1283 | 32 | 2026-03-30 15:07:29 | 2012-01-30 19:03:59 |
| niklastr_promise | 940 | 13 | 2025-04-21 12:32:33 | 2017-08-21 20:19:19 |
| sinzlab_nndichromacy | 467 | 13 | 2024-09-03 22:24:47 | 2020-05-16 21:55:16 |
| ubarsc_python-fmask | 372 | 16 | 2025-03-26 04:37:47 | 2015-09-15 06:34:54 |


In [27]:
# Create the final CSV in the exact required format

yournetid = "pmarino"

final_df = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

# Rename columns exactly as required
final_df.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

# Save as semicolon-separated CSV
final_df.to_csv(
    yournetid + '_project_summary.csv',
    sep=';',
    index=False
)

final_df.head()

,project_wocid,commit_sha1,author,time,commit message
0,ubarsc_python-fmask,00489059106ca30896d323bbd07cb9600f50e890,Neil Flood <neilflood@fastmail.fm>,1643078770,fillminima.py flake8 errors\n
1,ubarsc_python-fmask,01ce91861d7887fd9954ba3646ec4ce8d6d7a796,Sam Gillingham <gillingham.sam@gmail.com>,1461733841,fixed warning on OSX where I was treating an i...
2,ubarsc_python-fmask,01d87f03862865585a61c845591d95037462ce81,Neil Flood <neilflood@fastmail.fm>,1725316082,Add explicit instructions for setting the cond...
3,ubarsc_python-fmask,01edcf55b82564c9a9aaf43e514c93e0f9baa654,Neil Flood <Neil.Flood@derm.qld.gov.au>,1464912459,Added framework to set up a filter on small cl...
4,ubarsc_python-fmask,039a848a80a1049c11656a6e3fd30154c6e27b10,Sam Gillingham <gillingham.sam@gmail.com>,1662357199,update Neil's email and add MANIFEST.in (#62)\n\n
